### 1. Character Level Trigram Language model

In [3]:
import torch
import urllib.request
import re

# Fetch & Download Berkeley Restaurant Project Data
print("Fetching transcript data")
url = "https://raw.githubusercontent.com/wooters/berp-trans/master/transcript.txt"
response = urllib.request.urlopen(url)
data = response.read().decode('utf-8')
print(data[:100])

Fetching transcript data
33_1_0001 okay let's see i want to go to a thai restaurant . [uh] with less than ten dollars per per


In [4]:
# remove IDs, keep only letters and spaces
cleaned_lines = []
sentences = [line.split(" ", 1)[1] for line in data.strip().split('\n') if len(line.split(" ", 1)) > 1] # 8566

# join with a single space, then clean
clean_text = " . ".join(sentences).lower()
clean_text = re.sub(r'[^a-z .]', '', clean_text)
print(clean_text[:500])

okay lets see i want to go to a thai restaurant . uh with less than ten dollars per person . i like to eat uh i like to eat at lunch time . so that would be eleven am to one pm . i dont want to walk for more than five minutes . tell me more about the uh na nakapan uh restaurant on martin luther king . i like to go to a hamburger restaurant . lets start again . i like to get a hamburger at an american restaurant . id like to eat dinner . and i dont mind walking uh . for half an hour . i dont want


In [5]:
# Preprocessing 
chars = sorted(list(set(clean_text))) # set-remove duplicates, -> list -> sort

# counts = {char: full_text.count(char) for char in chars}
# sorted_data = dict(sorted(counts.items(), key=lambda item:item[1], reverse=True))
# print(sorted_data)

# Encoding
stoi = {s:i for i, s in enumerate(chars)}
itos = {i:s for i, s in enumerate(chars)}
vocab_size = len(chars) # 61'

In [6]:
# Trigram count matrix
N = torch.ones((vocab_size, vocab_size, vocab_size), dtype=torch.int32) *5 # [61x61] # Laplace Smoothing, more weight to rate transitions.

for i in range(len(clean_text)-2):
	ch1 = clean_text[i]
	ch2 = clean_text[i+1]
	ch3 = clean_text[i+2]

	ix1 = stoi[ch1]
	ix2 = stoi[ch2]
	ix3 = stoi[ch3]
	N[ix1][ix2][ix3] += 1

# Normalize into probability Matrix P
P = N.float()
P /= P.sum(dim=2, keepdim=True)  # [28, 28, 28]

In [7]:
def generate(char1, char2, length=100):
	result = [char1, char2]
	
	for _ in range(length):
		ix1 = stoi[char1]
		ix2 = stoi[char2]
		
		temperature = 0.5
		probs = P[ix1, ix2]
		probs = probs ** (1.0 / temperature)
		probs = probs / probs.sum() # re-normalize
		next_idx = torch.multinomial(probs, num_samples=1, replacement=True)[0].item()
		
		char3 = itos[next_idx]
		# if char3 == '.': # Stop generation at sentence end
		# 	break
			
		result.append(char3)
		char1, char2 = char2, char3
	
	return "".join(result)

In [11]:
gen1 = generate("i", " ", length=100)
gen2 = generate("a", "r", length=100)
gen3 = generate("r", "e", length=100)

print(gen1)
print(gen2)
print(gen3)

i want to the bres . i wants . i wanch . to than food like thinnexpen i wan to eat pent . is . i would
art thain you te . stauran to fore i whe som to to food . ind lood like ther . i wan yout to eapk i wa
re mer . ise . ind be the . noistaurant . inut to eat . likell me re ve to i wan doll me an fore to to


### 2. Word level Language Model

In [12]:
words = clean_text.split()
unique_words = sorted(list(set(words))) 

stoi = {s:i for i, s in enumerate(unique_words)}
itos = {i:s for i, s in enumerate(unique_words)}
vocab_size = len(unique_words) # 1603

In [13]:
N = torch.ones((vocab_size, vocab_size), dtype=torch.int32)

for i in range(len(words)-1):
	w1, w2 = words[i], words[i+1]
	ix1, ix2 = stoi[w1], stoi[w2]
	N[ix1, ix2] += 1

P = N.float()
P /= P.sum(dim=1, keepdim=True)

In [17]:
def generate_word_level(start_word, length=10):
	current_idx = stoi[start_word]
	result = [start_word]

	for _ in range(length):
		probs = P[current_idx]
		next_idx = torch.multinomial(probs, num_samples=1)[0].item()

		word = itos[next_idx]
		result.append(word)

		current_idx = next_idx
	return " ".join(result)

In [27]:
generate_word_level("i")

'i apple japanese touristy cappuccino meters prompt rather hi weeknight fruits'

In [29]:
words = clean_text.split()
unique_words = sorted(list(set(words))) 

stoi = {s:i for i, s in enumerate(unique_words)}
itos = {i:s for i, s in enumerate(unique_words)}


In [30]:
# define 3D matrix
vocab_size = len(unique_words) # 1603
N = torch.ones((vocab_size, vocab_size, vocab_size), dtype=torch.int32)

# populate matrix
for i in range(len(words) - 2):
    w1, w2, w3 = words[i], words[i+1], words[i+2]
    ix1, ix2, ix3 = stoi[w1], stoi[w2], stoi[w3]
    N[ix1, ix2, ix3] += 1

# Normalize to probabilities
P = N.float()
P /= P.sum(dim=2, keepdim=True)

def generate_word_trigram(word1, word2, length=10):
    result = [word1, word2]
    
    for _ in range(length):
        ix1, ix2 = stoi[word1], stoi[word2]
        probs = P[ix1, ix2]
        
        next_idx = torch.multinomial(probs, num_samples=1)[0].item()
        next_word = itos[next_idx]
        
        if next_word == '.':
            break
            
        result.append(next_word)
        # Shift context forward
        word1, word2 = word2, next_word
        
    return " ".join(result)

In [ ]:
print(generate_word_trigram("i", "want", 10))
print(generate_word_trigram("i", "would", 10))
print(generate_word_trigram("is", "there", 10))

i want eighteen lalimes twen hongkongeastocean fiesta should salsa whoops product fettucini
i would like national joint junior database irrelevant paninis margaritas bay other
is there dammit visit lillys accompany so healthy under coupla heike later
